In [1]:
import numpy
import pandas
import yfinance

ativo = "BOVA11.SA"
inicio = "2023-01-01"

df = yfinance.download(ativo, start=inicio)['Close']
df_retornos = numpy.log(df / df.shift(1)).dropna()

x = df_retornos.values.flatten()

[*********************100%***********************]  1 of 1 completed


In [2]:
num_lags = 3
N = len(x) - num_lags
y = (x[num_lags:] > 0).astype(float)

X_list = []
for i in range(N):
    lags = x[i : i + num_lags]
    X_list.append(numpy.insert(lags, 0, 1.0))

X = numpy.array(X_list)

In [3]:
def sigmoide(z):
    z_limitado = numpy.clip(z, -500, 500)
    return 1.0 / (1.0 + numpy.exp(-z_limitado))

In [4]:
num_atributos = X.shape[1]
beta = numpy.zeros(num_atributos)

max_iteracoes = 20
tolerancia = 1e-6

for iteracao in range(max_iteracoes):
    z = X @ beta
    p = sigmoide(z)
    gradiente = X.T @ (p - y)
    w = p * (1.0 - p)
    w = numpy.maximum(w, 1e-5)
    hessiana = X.T @ (X * w[:, numpy.newaxis])

    delta_beta = numpy.linalg.inv(hessiana) @ gradiente
    beta = beta - delta_beta
    
    norma_passo = numpy.linalg.norm(delta_beta)
    print(f"Iteração {iteracao + 1}: ||delta_beta|| = {norma_passo:.6f}")
    
    if norma_passo < tolerancia:
        break

Iteração 1: ||delta_beta|| = 10.690170
Iteração 2: ||delta_beta|| = 0.042652
Iteração 3: ||delta_beta|| = 0.000002
Iteração 4: ||delta_beta|| = 0.000000


In [5]:
probabilidades_finais = sigmoide(X @ beta)
previsoes = (probabilidades_finais >= 0.5).astype(float)
acuracia = numpy.mean(previsoes == y)

print("\n" + "="*40)
print(f"Beta_0 (Intercepto): {beta[0]:.4f}")
for i in range(1, len(beta)):
    print(f"Beta_{i} (Lag {i}):        {beta[i]:.4f}")

print(f"\nAcurácia no treino: {acuracia * 100:.2f}%")


Beta_0 (Intercepto): 0.0520
Beta_1 (Lag 1):        8.6034
Beta_2 (Lag 2):        1.5094
Beta_3 (Lag 3):        -6.2364

Acurácia no treino: 53.42%
